<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 10px 0; color:#cba6f7; font-weight:bold; font-size:1.05em;">📎 Session 2D — Dict (Deep) · Hands-on</p>
  <p style="margin:0;">The runnable playground for the dict theory — construction, the method surface, comprehensions, <code>Counter</code>/<code>defaultdict</code>, the seven edge cases, and the ML patterns. A dict is a <strong style="color:#89b4fa">hash table mapping hashable keys to arbitrary values</strong> — O(1) access plus insertion order.</p>
</div>

### 1. O(1) key access

In [1]:
big = {i: i for i in range(1_000_000)}
big[999999]        # one hash lookup - O(1), no scan

999999

In [2]:
# vs scanning a list of (key, value) pairs for the same key - O(n)
pairs = list(big.items())
def scan_lookup(pairs, key):
    for k, v in pairs:
        if k == key:
            return v
scan_lookup(pairs, 999999)

999999

In [3]:
import time
start = time.perf_counter(); big[999999];            t_dict = time.perf_counter() - start
start = time.perf_counter(); scan_lookup(pairs, 999999); t_scan = time.perf_counter() - start
print(f'dict lookup: {t_dict*1e6:8.2f} us')
print(f'list scan:   {t_scan*1e6:8.2f} us')

dict lookup:    72.00 us
list scan:   41008.60 us


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0;">A dict hashes the key, jumps straight to the bucket, and returns the value — constant time no matter how big the dict is. Scanning a list of pairs is O(n). <strong style="color:#cba6f7">SQL anchor:</strong> a dict is a lookup table with a primary-key index; <code>d[key]</code> is an indexed seek, the scan is a full table scan.</p>
</div>

### 2. Construction — the ways to build a dict

In [4]:
{"a": 1, "b": 2}                      # literal

{'a': 1, 'b': 2}

In [5]:
dict(a=1, b=2)                        # keyword args (string keys only)

{'a': 1, 'b': 2}

In [6]:
dict([("a", 1), ("b", 2)])            # from an iterable of pairs

{'a': 1, 'b': 2}

In [7]:
dict(zip(["a", "b", "c"], [1, 2, 3])) # zip two parallel lists

{'a': 1, 'b': 2, 'c': 3}

In [8]:
dict.fromkeys(["x", "y", "z"], 0)     # same value for every key

{'x': 0, 'y': 0, 'z': 0}

In [9]:
{x: x * x for x in range(5)}          # comprehension

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}

### 3. Access — `[]` vs `.get()`

In [10]:
d = {"a": 1, "b": 2}
d["a"]                # 1

1

In [11]:
try:
    d["z"]            # KeyError - the key must exist
except KeyError as e:
    print("KeyError:", e)

KeyError: 'z'


In [12]:
print(d.get("z"))         # None - safe, no crash
print(d.get("z", 0))      # 0   - with a fallback default

None
0


<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0;">Use <code>d[key]</code> when the key <strong style="color:#89b4fa">must exist</strong> (fail loud on a bug). Use <code>d.get(key, default)</code> when absence is a <strong style="color:#a6e3a1">normal, expected case</strong> with a sensible fallback.</p>
</div>

### 4. The method surface

In [13]:
d = {"a": 1, "b": 2}
d.setdefault("c", 3)      # inserts c=3 (missing) and returns 3

3

In [14]:
d.setdefault("a", 99)     # a exists -> returns existing 1, no overwrite

1

In [15]:
d

{'a': 1, 'b': 2, 'c': 3}

In [16]:
list(d.keys()), list(d.values()), list(d.items())

(['a', 'b', 'c'], [1, 2, 3], [('a', 1), ('b', 2), ('c', 3)])

In [17]:
d.update({"b": 20, "d": 4})   # merge / overwrite in place
d

{'a': 1, 'b': 20, 'c': 3, 'd': 4}

In [18]:
d.pop("d")                # removes "d", RETURNS its value 4

4

In [19]:
d.popitem()               # removes & RETURNS the last-inserted (key, value)

('c', 3)

In [20]:
d

{'a': 1, 'b': 20}

In [21]:
del d["a"]                # delete by key (returns nothing)
d

{'b': 20}

In [22]:
"b" in d, "zzz" in d      # membership tests KEYS

(True, False)

<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 12px 0;">Which methods hand you a value vs only mutate:</p>
  <ul style="margin:0; padding-left:20px; line-height:2.2">
    <li><strong style="color:#a6e3a1">Return something useful</strong> — <code>get</code>, <code>pop</code>, <code>popitem</code>, <code>setdefault</code></li>
    <li><strong style="color:#f38ba8">Mutate, return None</strong> — <code>update</code>, <code>clear</code>; <code>del</code> is a statement</li>
  </ul>
</div>

### 5. Iteration — keys by default, `.items()` for pairs

In [23]:
d = {"a": 1, "b": 2, "c": 3}
for k in d:               # iterating a dict yields KEYS
    print(k, end=" ")

a b c 

In [24]:
for v in d.values():
    print(v, end=" ")

1 2 3 

In [25]:
for k, v in d.items():    # (key, value) pairs - the common loop
    print(f"{k}={v}", end="  ")

a=1  b=2  c=3  

In [26]:
try:
    for k, v in d:        # ValueError - unpacking a single key into two names
        pass
except ValueError as e:
    print("ValueError:", e)

ValueError: not enough values to unpack (expected 2, got 1)


### 6. Comprehensions — build / filter / invert

In [27]:
{x: x * x for x in range(5)}                     # build

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}

In [28]:
prices = {"apple": 3, "banana": 1, "cherry": 5}
{k: v for k, v in prices.items() if v > 2}       # filter  (WHERE v > 2)

{'apple': 3, 'cherry': 5}

In [29]:
{v: k for k, v in prices.items()}               # invert (values must be unique)

{3: 'apple', 1: 'banana', 5: 'cherry'}

<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0;">Inverting is handy for reverse lookups, but it is <strong style="color:#f38ba8">lossy when values are not unique</strong> — duplicate values collapse because keys must be unique (last one wins).</p>
</div>

### 7. Counter — counting made trivial

In [30]:
words = "the cat the dog the cat".split()
# the mechanism first, with .get(k, 0) + 1
f = {}
for w in words:
    f[w] = f.get(w, 0) + 1
f

{'the': 3, 'cat': 2, 'dog': 1}

In [31]:
from collections import Counter
c = Counter(words)        # the one-liner
c

Counter({'the': 3, 'cat': 2, 'dog': 1})

In [32]:
c.most_common(2)          # top 2 by count

[('the', 3), ('cat', 2)]

In [33]:
Counter("mississippi")    # counts characters

Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})

In [34]:
Counter("aab") + Counter("bcc")   # Counter arithmetic

Counter({'a': 2, 'b': 2, 'c': 2})

### 8. defaultdict — auto-create the missing value

In [35]:
from collections import defaultdict
people = [("eng", "Alice"), ("sales", "Bob"), ("eng", "Cara"), ("sales", "Dan")]
groups = defaultdict(list)
for dept, name in people:
    groups[dept].append(name)     # no KeyError on first touch
dict(groups)

{'eng': ['Alice', 'Cara'], 'sales': ['Bob', 'Dan']}

In [36]:
counts = defaultdict(int)
for w in words:
    counts[w] += 1                # missing key auto-starts at 0
dict(counts)

{'the': 3, 'cat': 2, 'dog': 1}

### 9. Merging dicts — right side wins

In [37]:
DEFAULTS = {"lr": 0.01, "epochs": 100, "batch": 32}
user = {"lr": 0.001, "batch": 64}
DEFAULTS | user                     # 3.9+  -> user overrides, epochs falls back

{'lr': 0.001, 'epochs': 100, 'batch': 64}

In [38]:
{**DEFAULTS, **user}                # older syntax, same result

{'lr': 0.001, 'epochs': 100, 'batch': 64}

In [39]:
merged = DEFAULTS.copy()
merged.update(user)                 # mutate a copy in place
merged

{'lr': 0.001, 'epochs': 100, 'batch': 64}

### Edge Cases (the seven from Chunk C)

In [40]:
{"a": 1, "a": 2, "a": 3}            # Edge 3: duplicate keys collapse -> {'a': 3}

{'a': 3}

In [41]:
dict([("x", 1), ("x", 2)])          # first value lost silently -> {'x': 2}

{'x': 2}

In [42]:
# Edge 4: dict.fromkeys with a mutable default shares ONE object
d = dict.fromkeys(["a", "b", "c"], [])
d["a"].append(1)
print(d)                                  # {'a': [1], 'b': [1], 'c': [1]}  -- all changed!

{'a': [1], 'b': [1], 'c': [1]}


In [43]:
d2 = {k: [] for k in ["a", "b", "c"]}     # comprehension -> distinct lists
d2["a"].append(1)
print(d2)                                 # {'a': [1], 'b': [], 'c': []}  -- correct

{'a': [1], 'b': [], 'c': []}


In [44]:
# Edge 5: modifying a dict during iteration raises
d = {"a": 1, "b": 2, "c": 3}
try:
    for k in d:
        if d[k] == 2:
            del d[k]        # RuntimeError: dictionary changed size during iteration
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: dictionary changed size during iteration


In [45]:
# fix: build a new dict with a comprehension (or iterate list(d))
d = {"a": 1, "b": 2, "c": 3}
{k: v for k, v in d.items() if v != 2}

{'a': 1, 'c': 3}

In [46]:
# Edge 6: 1 and 1.0 are the SAME key (equal, same hash); the string "1" is distinct
{1: "a", "1": "b", 1.0: "c"}            # -> {1: 'c', '1': 'b'}

{1: 'c', '1': 'b'}

In [47]:
d = {1: "a", "1": "b", 1.0: "c"}
"a" in d          # False -- "in" checks KEYS, not values

False

In [48]:
# Edge 7: .get() with a default does NOT store; setdefault does
d = {}
d.get("k", [])            # returns [] but leaves d unchanged
d

{}

In [49]:
d = {}
d.setdefault("k", []).append(1)   # stores [] AND returns it -> mutation sticks
d

{'k': [1]}

In [50]:
# bonus: unhashable keys are rejected
try:
    {[1, 2]: "x"}         # a list cannot be a key
except TypeError as e:
    print("TypeError:", e)

TypeError: unhashable type: 'list'


### ML Real-World Uses

In [51]:
# A feature vector / data row is a dict
sample = {"age": 34, "income": 55000, "region": "south", "score": 0.87}
print(sample["age"])
print(sample.get("missing_feat", 0))    # safe default for an absent feature

34
0


In [52]:
# Class distribution - step zero before training
labels = ["spam", "ham", "spam", "spam", "ham", "spam"]
dist = Counter(labels)
total = sum(dist.values())
balance = {k: round(v / total, 2) for k, v in dist.items()}
print(dist)
print(balance)

Counter({'spam': 4, 'ham': 2})
{'spam': 0.67, 'ham': 0.33}


In [53]:
# Vocabulary encoding - the tokenizer core
corpus = ["deploy docker", "docker build", "deploy kubernetes"]
vocab = {}
for doc in corpus:
    for tok in doc.split():
        if tok not in vocab:
            vocab[tok] = len(vocab)     # next token gets the next integer id
def encode(doc):
    return [vocab[t] for t in doc.split()]
print(vocab)
print(encode("deploy docker"))

{'deploy': 0, 'docker': 1, 'build': 2, 'kubernetes': 3}
[0, 1]


In [54]:
# Memoization - a dict cache turns exponential into linear
cache = {}
def fib(n):
    if n < 2:
        return n
    if n in cache:
        return cache[n]
    cache[n] = fib(n - 1) + fib(n - 2)
    return cache[n]
print(fib(30))
print(len(cache), "cached entries")

832040
29 cached entries


In [55]:
# Config merged over defaults - reproducible and immutable-friendly
DEFAULTS = {"lr": 0.01, "epochs": 100, "optimizer": "adam"}
run_config = DEFAULTS | {"lr": 0.001, "epochs": 50}
run_config

{'lr': 0.001, 'epochs': 50, 'optimizer': 'adam'}

### Worked Challenges

In [56]:
# C1 - invert a dict with NON-unique values (lossless, via defaultdict)
grades = {"Alice": 88, "Bob": 92, "Cara": 88}
inv = defaultdict(list)
for name, score in grades.items():
    inv[score].append(name)
dict(inv)                     # {88: ['Alice', 'Cara'], 92: ['Bob']}

{88: ['Alice', 'Cara'], 92: ['Bob']}

In [57]:
# C2 - group anagrams by their sorted-letter signature
def group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        key = "".join(sorted(w))    # canonical signature
        groups[key].append(w)
    return list(groups.values())
group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]

In [58]:
# C3 - two-sum in O(n) with a dict for O(1) complement lookup
def two_sum(nums, target):
    seen = {}
    for i, n in enumerate(nums):
        if target - n in seen:
            return [seen[target - n], i]
        seen[n] = i
    return None
two_sum([2, 7, 11, 15], 9)

[0, 1]

In [59]:
# C4 - top-k frequent elements via Counter.most_common
def top_k_frequent(items, k):
    return [x for x, _ in Counter(items).most_common(k)]
top_k_frequent([1, 1, 1, 2, 2, 3], 2)

[1, 2]

In [60]:
# C5 - first non-repeating element (the 2C set -> dict bridge)
def first_unique(seq):
    counts = Counter(seq)
    for x in seq:            # scan in ORIGINAL order
        if counts[x] == 1:
            return x
    return None
first_unique([2, 3, 2, 4, 3, 5])

4

<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0;">C5 is the seam where sets stop and dicts begin: a set knows only <strong style="color:#89b4fa">present/absent</strong>, but this needs <strong style="color:#a6e3a1">counts</strong> (how many times) <em>and</em> <strong style="color:#a6e3a1">order</strong> (which came first) — a <code>Counter</code> gives both. That is exactly why Dict follows Set in the track.</p>
</div>

<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8">
  <p style="margin:0 0 10px 0; color:#cba6f7; font-weight:bold;">🔑 Key Takeaways</p>
  <ul style="margin:0; padding-left:20px; line-height:2.2">
    <li>A dict is a hash table mapping <strong style="color:#89b4fa">hashable keys</strong> → arbitrary values; <code>d[key]</code> is O(1)</li>
    <li><code>[]</code> fails loud, <code>.get()</code> is safe</li>
    <li><code>Counter</code> to count, <code>defaultdict</code> to group</li>
    <li><code>fromkeys</code> with a mutable default aliases one object — use a comprehension</li>
    <li>Iterate <code>.items()</code> for pairs; never change a dict's size mid-iteration</li>
  </ul>
</div>